# Курсовая работа (проект)

## Тема: "Исследование котировок акций для обнаружения волатильных паттернов и рыночных трендов"

### Поиск и проверка датасетов

In [1]:
%pip install pandas requests beautifulsoup4

In [10]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup

In [28]:
date = '2025-09-01'  # любую дату, главное с закрытым торгом
base = 'https://iss.moex.com/iss/'

#### 3.1. Реляционный набор в CSV: история торгов и справочник бумаг

Источник - MOEX ISS. Обе исходные таблицы запрашиваются по адресам с расширением `.csv`, а затем сохраняются в локальные CSV-файлы. Загружаем все страницы за выбранную дату; многолетняя история будет собираться на следующем этапе.

CSV - способ записи таблицы. Реляционная структура определяется строками, атрибутами, ключами и связью между таблицами, а не расширением файла.

**Структура и связь таблиц**

- `securities` - справочник инструментов. Одна строка описывает инструмент; кандидат в первичный ключ - `secid`.
- `prices` - история торгов. Одна строка описывает инструмент в определённом режиме, за дату и торговую сессию. Проверяем составной ключ `SECID + BOARDID + TRADEDATE + TRADINGSESSION`.
- `prices.SECID` ссылается на `securities.secid`. Связь - много торговых наблюдений к одному инструменту.
- Проверяем отсутствие пустых и повторных ключей, наличие каждой бумаги истории в справочнике и сохранение количества строк после JOIN.

В аналитической таблице выбраны четыре категориальных поля (`SECID`, `BOARDID`, `CURRENCYID`, `type`) и семь числовых (`OPEN`, `HIGH`, `LOW`, `CLOSE`, `VOLUME`, `VALUE`, `NUMTRADES`). Справочник служит для связи и обогащения истории, а не отдельной выборкой цен. Тип из текущего справочника не трактуем как доказанный исторический атрибут.

In [ ]:
from pathlib import Path
from io import StringIO
from urllib.request import urlopen
from urllib.parse import urlencode

data_dir = Path('data') / 'relational_csv'
data_dir.mkdir(parents=True, exist_ok=True)
url = base + 'history/engines/stock/markets/shares/securities.csv'
url_securities = base + 'securities.csv'

def load_moex_csv(endpoint, block, params):
    pages = []
    start = 0
    for _ in range(1000):
        # Не отключаем iss.meta: в CSV это убирает заголовки столбцов.
        query = {**params, 'iss.only': block, 'start': start}
        with urlopen(endpoint + '?' + urlencode(query), timeout=30) as response:
            raw = response.read()
        try:
            text = raw.decode('utf-8-sig')
        except UnicodeDecodeError:
            text = raw.decode('cp1251')
        # Первые две строки ответа ISS - имя блока и пустая строка.
        frame = pd.read_csv(StringIO(text), sep=';', skiprows=2)
        if frame.empty:
            break
        pages.append(frame)
        start += len(frame)
    else:
        raise RuntimeError('Достигнут предел страниц; загрузка не завершена')
    if not pages:
        raise ValueError('Источник не вернул данных для выбранных параметров')
    return pd.concat(pages, ignore_index=True)

prices = load_moex_csv(url, 'history', {'date': date})

In [ ]:
history_key = ['SECID', 'BOARDID', 'TRADEDATE', 'TRADINGSESSION']
assert not prices[history_key].isna().any().any(), 'Пустые компоненты ключа истории'
assert not prices.duplicated(history_key).any(), 'Составной ключ истории не уникален'
prices.to_csv(data_dir / f'prices_{date}.csv', index=False, encoding='utf-8-sig')

In [ ]:
securities = load_moex_csv(
    url_securities, 'securities', {'engine': 'stock', 'market': 'shares'}
)
assert securities['secid'].notna().all(), 'Пустой ключ справочника'
assert securities['secid'].is_unique, 'Ключ справочника не уникален'
missing = set(prices['SECID']) - set(securities['secid'])
assert not missing, f'В справочнике отсутствуют бумаги: {sorted(missing)}'

stocks = prices.merge(
    securities[['secid', 'type']], left_on='SECID', right_on='secid',
    how='left', validate='many_to_one', indicator=True
)
assert stocks['_merge'].eq('both').all(), 'Есть строки без соответствия'
assert len(stocks) == len(prices), 'JOIN изменил число торговых наблюдений'
stocks = stocks[['SECID', 'BOARDID', 'CURRENCYID', 'type', 'TRADEDATE',
                 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME', 'VALUE', 'NUMTRADES']]
securities.to_csv(data_dir / 'securities.csv', index=False, encoding='utf-8-sig')
stocks.to_csv(data_dir / f'stocks_joined_{date}.csv', index=False, encoding='utf-8-sig')

In [ ]:
print('История торгов, строк:', len(prices))
print('Справочник, строк:', len(securities))
print('Ключ справочника secid уникален:', securities['secid'].is_unique)
print('Ключ истории:', ' + '.join(history_key))
print('Нет повторных ключей истории:', not prices.duplicated(history_key).any())
print('Бумаги без справочника:', len(missing))
print('JOIN сохранил число строк:', len(stocks) == len(prices))
print('CSV-файлы сохранены в:', data_dir.resolve())

In [ ]:
display(stocks.head(10))

#### 3.2. XML: история торгов в иерархическом формате

Структура: document -> data -> rows -> row. Первая страница ответа

Биржевые наблюдения в XML

In [ ]:
url_xml = base + 'history/engines/stock/markets/shares/securities.xml'

r = requests.get(url_xml, params={'date': date}, timeout=30)
r.raise_for_status()

In [ ]:
root = ET.fromstring(r.content)
rows = [row.attrib for row in root.findall("./data[@id='history']/rows/row")]
xml_data = pd.DataFrame(rows)[['SECID', 'BOARDID', 'CURRENCYID', 'TRADEDATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME']]

In [ ]:
display(xml_data.head(10))

#### 3.3. JSON API: индексы как возможные факторы влияния на акции

In [ ]:
url_json = base + 'history/engines/stock/markets/index/securities.json'

In [ ]:
parts = []

for start in range(0, 10000, 100):
    r = requests.get(url_json, params={'date': date, 'start': start, 'iss.meta': 'off'}, timeout=30)
    r.raise_for_status()
    data = r.json()['history']
    if not data['data']:
        break
    parts.append(pd.DataFrame(data['data'], columns=data['columns']))
else:
    raise ValueError('Нужно увеличить диапазон страниц индексов')

indices = pd.concat(parts, ignore_index=True)
indices = indices[['SECID', 'BOARDID', 'CURRENCYID', 'TRADEDATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE']]

In [ ]:
print('В ответе есть индексы и расчётные показатели фондов')
print('IMOEX и RTSI — рынок акций, RGBI — облигации, RVI — рыночная волатильность')

In [ ]:
display(indices[indices['SECID'].isin(['IMOEX', 'RTSI', 'RGBI', 'RVI'])])

#### 3.4. Текстовые данные: полные новости Банка России
RSS нужен для поиска ссылок, сами тексты получаем со страниц статей

In [ ]:
url_news = 'https://www.cbr.ru/rss/eventrss'

r = requests.get(url_news, timeout=30)
r.raise_for_status()
rss = ET.fromstring(r.content)

In [ ]:
rows = []

for item in rss.findall('./channel/item')[:10]:
    link = item.findtext('link')
    
    r = requests.get(link, timeout=30)
    r.raise_for_status()
    
    soup = BeautifulSoup(r.content, 'html.parser')
    blocks = soup.select('.lead-text, .landing-text, .full-text')
    text = ' '.join(block.get_text(' ', strip=True) for block in blocks)
    
    rows.append({'title': item.findtext('title'), 'date': item.findtext('pubDate'), 'url': link, 'text': text})

news = pd.DataFrame(rows)

Простая разметка: тема, аудитория и наличие слов о росте/снижении

In [ ]:
news['topic'] = 'другое'

news.loc[news['text'].str.contains('инфляц|ставк', case=False), 'topic'] = 'ставки и инфляция'
news.loc[news['text'].str.contains('валют|курс рубл', case=False), 'topic'] = 'валюты'

news['audience'] = 'общая'

news.loc[news['text'].str.contains('предприят|компани', case=False), 'audience'] = 'бизнес'
news.loc[news['text'].str.contains('граждан|населен', case=False), 'audience'] = 'население'

news['direction'] = 'не определено'

news.loc[news['text'].str.contains('рост|повыш', case=False), 'direction'] = 'есть рост'
news.loc[news['text'].str.contains('сниж|сокращ', case=False), 'direction'] = 'есть снижение'

news['characters'] = news['text'].str.len()
news['words'] = news['text'].str.findall(r'\b[А-Яа-яЁёA-Za-z]+\b').str.len()
news['numbers'] = news['text'].str.findall(r'\d+(?:[.,]\d+)?').str.len()

In [ ]:
print('Фактор: макроэкономические сообщения. Категории и числа вычислены из текста')
print('Разметка приблизительная: *direction* не означает влияние новости на цену акции')

In [ ]:
display(news[['title', 'topic', 'audience', 'direction', 'characters', 'words', 'numbers']].head(8))

#### Характеристики, которые выбраны для каждого набора

In [ ]:
datasets = {'Акции': stocks, 'XML': xml_data, 'Индексы JSON': indices, 'Новости': news}

categories = {
    'Акции': ['SECID', 'BOARDID', 'CURRENCYID', 'type'],
    'XML': ['SECID', 'BOARDID', 'CURRENCYID'],
    'Индексы JSON': ['SECID', 'BOARDID', 'CURRENCYID'],
    'Новости': ['topic', 'audience', 'direction']}

numbers = {
    'Акции': ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME', 'VALUE', 'NUMTRADES'],
    'XML': ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME'],
    'Индексы JSON': ['OPEN', 'HIGH', 'LOW', 'CLOSE'],
    'Новости': ['characters', 'words', 'numbers']}

meaning = {
    'SECID': 'Код бумаги или индекса', 'BOARDID': 'Режим торгов',
    'CURRENCYID': 'Валюта расчётов', 'type': 'Тип бумаги из текущего справочника',
    'OPEN': 'Цена открытия / начальное значение индекса',
    'HIGH': 'Максимум за день', 'LOW': 'Минимум за день',
    'CLOSE': 'Цена / значение закрытия', 'VOLUME': 'Объём торгов',
    'VALUE': 'Денежный оборот в валюте расчётов', 'NUMTRADES': 'Количество сделок',
    'TRADEDATE': 'Дата торгов', 'date': 'Время публикации с часовым поясом',
    'title': 'Заголовок', 'text': 'Полный текст', 'url': 'Ссылка на статью',
    'topic': 'Тема по ключевым словам', 'audience': 'Предполагаемая аудитория',
    'direction': 'Упомянутое изменение; при двух совпадениях выбранное снижение',
    'characters': 'Количество символов', 'words': 'Количество слов',
    'numbers': 'Количество упоминаний чисел'}

In [ ]:
display(pd.DataFrame(meaning.items(), columns=['Поле', 'Значение']))

#### Итоговое описание всех датасетов

In [ ]:
def describe_data(name, df):
    
    df.replace('', pd.NA, inplace=True)
    
    for col in numbers[name]:
        df[col] = pd.to_numeric(df[col], errors='raise')
    
    for col in categories[name]:
        df[col] = df[col].astype('category')
    
    date_col = 'date' if name == 'Новости' else 'TRADEDATE'
    df[date_col] = pd.to_datetime(df[date_col], utc=True, errors='raise')
    
    print('\n', name, '— строк:', len(df), 'столбцов:', len(df.columns))
    print('Категориальные:', categories[name])
    print('Числовые:', numbers[name])
    print('Диапазон дат:', df[date_col].min(), '—', df[date_col].max())
    display(pd.DataFrame({'Тип': df.dtypes.astype(str), 'Пропуски': df.isna().sum(), 'Пропуски, %': (df.isna().mean()*100).round(2), 'Уникальные': df.nunique()}))
    display(df[numbers[name]].agg(['min', 'max']).T)
    print('Полных дубликатов:', df.duplicated().sum())

In [ ]:
for name, df in datasets.items():
    describe_data(name, df)

Проверки качества данных

In [ ]:
checks = []

for name, df in datasets.items():
    checks.append({'Проверка': name + ': минимум 3 категории и 3 числа', 'Пройдено': len(categories[name]) >= 3 and len(numbers[name]) >= 3})
    checks.append({'Проверка': name + ': нет полных дубликатов', 'Пройдено': df.duplicated().sum() == 0})

for name in ['Акции', 'XML', 'Индексы JSON']:
    data = datasets[name][['OPEN', 'HIGH', 'LOW', 'CLOSE']].dropna()
    correct = (data['LOW'] <= data['OPEN']) & (data['OPEN'] <= data['HIGH'])
    correct = correct & (data['LOW'] <= data['CLOSE']) & (data['CLOSE'] <= data['HIGH'])
    checks.append({'Проверка': name + ': Low ≤ Open, Close ≤ High', 'Пройдено': bool(correct.all())})

checks.append({'Проверка': 'JOIN сохранил число строк', 'Пройдено': len(stocks) == len(prices)})
checks.append({'Проверка': 'Типы бумаг найдены', 'Пройдено': not stocks['type'].isna().any()})
checks.append({'Проверка': 'Объёмы неотрицательны', 'Пройдено': bool(stocks['VOLUME'].dropna().ge(0).all())})
checks.append({'Проверка': 'Тексты статей извлечены', 'Пройдено': bool(news['characters'].ge(100).all())})
checks.append({'Проверка': 'Нет повторных ссылок новостей', 'Пройдено': news['url'].is_unique})

In [ ]:
print('Пропуски цен не заполняем нулями: они могут быть связаны с отсутствием торгов')
print('Новости из текущей RSS и исторические котировки пока не объединяем: даты различаются')
print('У текстов условие 3+3 выполнено после обработки, а не по исходным готовым полям')

In [ ]:
display(pd.DataFrame(checks))

In [ ]:
print('Доля пройденных проверок:', round(pd.DataFrame(checks)['Пройдено'].mean()*100, 1), '%')

### Ссылки на использованные источники

In [ ]:
sources = pd.DataFrame({
    'Источник': ['История акций CSV', 'Справочник CSV', 'XML', 'Индексы JSON', 'Новости ЦБ', 'MOEX ISS'],
    'Ссылка': [url + '?' + urlencode({'date': date, 'iss.only': 'history'}),
               url_securities + '?' + urlencode({'engine': 'stock', 'market': 'shares', 'iss.only': 'securities'}),
               url_xml + '?date=' + date, url_json + '?date=' + date,
               url_news, 'https://www.moex.com/a2920']})

In [ ]:
for _, row in sources.iterrows():
    print(row['Источник'] + ': ' + row['Ссылка'])